# 01 - Data Access Check

Verify the supplied archive, CSV contract and filename mappings only. No image contents are opened. Sample-submission classes are placeholders, never test labels.

## 1. Project root and kernel

Use the same structural root detection as the environment notebook. Run from the root or `notebooks/` with the project kernel.

In [1]:
from pathlib import Path, PurePosixPath, PureWindowsPath
from collections import Counter, defaultdict
from datetime import datetime
import json
import re
import stat
import sys
import zipfile
import pandas as pd

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks" / "00_Environment_Check.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this project root or its notebooks directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
assert find_project_root(PROJECT_ROOT / "notebooks") == PROJECT_ROOT
assert find_project_root(PROJECT_ROOT) == PROJECT_ROOT
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve()
assert Path(sys.executable).resolve().parent == PROJECT_ROOT / ".venv" / "Scripts"
RAW = PROJECT_ROOT / "data/raw"
SUMMARY_PATH = PROJECT_ROOT / "outputs/metrics/data_access_check.json"
checks = {}
failed_checks = []

def check(name, condition):
    checks[name] = bool(condition)
    if not condition:
        failed_checks.append(name)
    print(f"{name}: {'PASS' if condition else 'FAIL'}")

def relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()

print("Working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


Working directory: C:\Users\leoca\Kaggle-Competition\notebooks
Project root: C:\Users\leoca\Kaggle-Competition
Python: C:\Users\leoca\Kaggle-Competition\.venv\Scripts\python.exe


## 2. Archive structure and extracted files

Extraction was preflighted before writing: unsafe paths, links and conflicts were rejected. Files were created exclusively, preserving nested paths. An existing identical ZIP in `data/raw/` was retained, as was the Downloads original. This notebook checks access to the extracted data; it does not extract again or open images.

In [2]:
ARCHIVE_NAME = "enge-707-2026-s-2-cat-and-dog-facial-feature-classification-ch.zip"
# Prefer the existing local archive; do not copy or download another archive.
archive_path = RAW / ARCHIVE_NAME
if not archive_path.is_file():
    archive_path = Path.home() / "Downloads" / ARCHIVE_NAME

with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    unsafe = []
    member_keys = []
    for member in members:
        name = member.filename.replace("\\", "/")
        path = PurePosixPath(name)
        mode = member.external_attr >> 16
        if (path.is_absolute() or PureWindowsPath(name).drive
                or not path.parts or ".." in path.parts
                or any(":" in part for part in path.parts)
                or stat.S_ISLNK(mode)
                or stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)):
            unsafe.append(member.filename)
        member_keys.append(name.casefold())
    archive_files = [m for m in members if not m.is_dir()]
    archive_names = {m.filename.replace("\\", "/") for m in archive_files}
    archive_directories = dict(sorted(Counter(str(PurePosixPath(m.filename).parent) for m in archive_files).items()))
check("archive_paths_safe_no_links", not unsafe)
check("archive_paths_unique_case_insensitive", len(member_keys) == len(set(member_keys)))

# Filesystem names and metadata only; no image decoding or pixel access.
files = [p for p in RAW.rglob("*") if p.is_file()]
extracted_names = {p.relative_to(RAW).as_posix() for p in files}
check("all_archive_files_extracted", archive_names <= extracted_names)
archive_summary = {
    "path": str(archive_path),
    "file_count": len(archive_files),
    "uncompressed_bytes": sum(m.file_size for m in archive_files),
    "files_by_directory": archive_directories,
    "unsafe_members": unsafe[:10],
    "extraction_policy": "Preserve structure; exclusive creation; skip only identical existing files; reject conflicts and links.",
    "original_and_existing_local_zip_preserved": (Path.home() / "Downloads" / ARCHIVE_NAME).is_file() and (RAW / ARCHIVE_NAME).is_file(),
}
print(json.dumps(archive_directories, indent=2))


archive_paths_safe_no_links: PASS
archive_paths_unique_case_insensitive: PASS


all_archive_files_extracted: PASS
{
  ".": 2,
  "test/test": 29443,
  "train/train": 63660
}


## 3. CSV schema and exact strings

Read every field as a string with automatic NA conversion disabled. Preserve IDs, class spellings and row order. Empty or whitespace-only fields are reported as missing without modifying them. Only the training CSV supplies class names.

In [3]:
def unique_file(name):
    matches = [p for p in files if p.name == name]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one {name}; found {matches}")
    return matches[0]

train_csv = unique_file("train_labels.csv")
sample_csv = unique_file("sample_submission.csv")
train = pd.read_csv(train_csv, dtype=str, keep_default_na=False, na_filter=False)
sample = pd.read_csv(sample_csv, dtype=str, keep_default_na=False, na_filter=False)

def describe_csv(frame, path):
    return {
        "path": relative(path),
        "columns": list(frame.columns),
        "row_count": len(frame),
        "missing_values": {column: int(frame[column].str.strip().eq("").sum()) for column in frame.columns},
        "duplicate_id_rows": int(frame["image_id"].duplicated().sum()),
        "all_rows_with_duplicate_ids": int(frame["image_id"].duplicated(keep=False).sum()),
    }

check("train_columns", list(train.columns) == ["image_id", "class"])
check("sample_columns", list(sample.columns) == ["image_id", "class"])
# Stop clearly on an incompatible schema rather than guessing ID columns.
if "image_id" not in train or "class" not in train or "image_id" not in sample or "class" not in sample:
    SUMMARY_PATH.write_text(json.dumps({"checks": checks, "failed_checks": failed_checks}, indent=2) + "\n", encoding="utf-8")
    raise ValueError("Expected image_id and class columns; see saved failed checks.")

csv_summary = {"train": describe_csv(train, train_csv), "sample_submission": describe_csv(sample, sample_csv)}
for name, description in csv_summary.items():
    check(f"{name}_no_missing_values", not any(description["missing_values"].values()))
    check(f"{name}_unique_ids", description["duplicate_id_rows"] == 0)
classes = sorted(train["class"].unique().tolist())
check("eight_training_classes", len(classes) == 8)
print(json.dumps(csv_summary, indent=2))
print("Exact training classes:", classes)
# Do not calculate unique values or a class distribution for sample placeholders.


train_columns: PASS
sample_columns: PASS
train_no_missing_values: PASS
train_unique_ids: PASS
sample_submission_no_missing_values: PASS
sample_submission_unique_ids: PASS
eight_training_classes: PASS
{
  "train": {
    "path": "data/raw/train_labels.csv",
    "columns": [
      "image_id",
      "class"
    ],
    "row_count": 63660,
    "missing_values": {
      "image_id": 0,
      "class": 0
    },
    "duplicate_id_rows": 0,
    "all_rows_with_duplicate_ids": 0
  },
  "sample_submission": {
    "path": "data/raw/sample_submission.csv",
    "columns": [
      "image_id",
      "class"
    ],
    "row_count": 29443,
    "missing_values": {
      "image_id": 0,
      "class": 0
    },
    "duplicate_id_rows": 0,
    "all_rows_with_duplicate_ids": 0
  }
}
Exact training classes: ['CatEye', 'CatFace', 'CatMouth', 'CatNose', 'DogEye', 'DogFace', 'DogMouth', 'DogNose']


## 4. Image directories and ID mappings

Verify the observed convention: training IDs are five decimal digits and match `<ID>.jpg` exactly. Submission IDs are unpadded positive decimal strings and map to `<ID padded to five digits>.jpg`. The source IDs remain unchanged. Indexing all image basenames detects ambiguous matches, and coverage checks detect unused images or many-to-one mappings.

In [4]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
images = [p for p in files if p.suffix.lower() in IMAGE_SUFFIXES]

def locate_images(split):
    candidates = [p for p in images if split in p.relative_to(RAW).parts[:-1]]
    directories = sorted({p.parent for p in candidates})
    if len(directories) != 1:
        raise ValueError(f"Expected one image-containing {split} directory; found {directories}")
    return directories[0], candidates

train_directory, train_images = locate_images("train")
test_directory, test_images = locate_images("test")
check("train_filename_convention", all(re.fullmatch(r"[0-9]{5}\.jpg", p.name) for p in train_images))
check("test_filename_convention", all(re.fullmatch(r"[0-9]{5}\.jpg", p.name) for p in test_images))
check("training_ids_five_digits", bool(train["image_id"].str.fullmatch(r"[0-9]{5}").all()))
check("submission_ids_unpadded_positive_decimals", bool(sample["image_id"].str.fullmatch(r"[1-9][0-9]*").all()))
check("train_test_directories_distinct", train_directory != test_directory)

def verify_mapping(frame, image_paths, pad_ids):
    by_name = defaultdict(list)
    for path in image_paths:
        by_name[path.name].append(path)
    missing, ambiguous, mapped = [], [], []
    examples = []
    # Iteration follows the original CSV row order; IDs are never rewritten.
    for original_id in frame["image_id"]:
        filename = (original_id.zfill(5) if pad_ids else original_id) + ".jpg"
        matches = by_name.get(filename, [])
        if not matches:
            missing.append(original_id)
        elif len(matches) != 1:
            ambiguous.append(original_id)
        else:
            mapped.append(matches[0])
            if len(examples) < 3:
                examples.append({"original_id": original_id, "filename": filename})
    reused = sum(count - 1 for count in Counter(mapped).values() if count > 1)
    unused = sorted(set(image_paths) - set(mapped))
    return {
        "rows_checked": len(frame),
        "matched_rows": len(mapped),
        "missing_count": len(missing), "missing_examples": missing[:10],
        "ambiguous_count": len(ambiguous), "ambiguous_examples": ambiguous[:10],
        "many_to_one_mapping_count": reused,
        "unused_image_count": len(unused),
        "unused_image_examples": [relative(p) for p in unused[:10]],
        "examples_in_original_row_order": examples,
    }

mapping = {
    "train": verify_mapping(train, train_images, pad_ids=False),
    "sample_submission": verify_mapping(sample, test_images, pad_ids=True),
}
for split, result in mapping.items():
    check(f"{split}_every_row_exactly_one_image", result["matched_rows"] == result["rows_checked"] and result["ambiguous_count"] == 0)
    check(f"{split}_one_to_one_full_image_coverage", result["many_to_one_mapping_count"] == 0 and result["unused_image_count"] == 0)
check("training_row_image_counts_equal", len(train) == len(train_images))
check("submission_row_test_image_counts_equal", len(sample) == len(test_images))
check("all_images_accounted_for", len(images) == len(train_images) + len(test_images))
print("Training image directory:", relative(train_directory), "files:", len(train_images))
print("Test image directory:", relative(test_directory), "files:", len(test_images))
print(json.dumps(mapping, indent=2))


train_filename_convention: PASS
test_filename_convention: PASS
training_ids_five_digits: PASS
submission_ids_unpadded_positive_decimals: PASS
train_test_directories_distinct: PASS


train_every_row_exactly_one_image: PASS
train_one_to_one_full_image_coverage: PASS
sample_submission_every_row_exactly_one_image: PASS
sample_submission_one_to_one_full_image_coverage: PASS
training_row_image_counts_equal: PASS
submission_row_test_image_counts_equal: PASS
all_images_accounted_for: PASS
Training image directory: data/raw/train/train files: 63660
Test image directory: data/raw/test/test files: 29443
{
  "train": {
    "rows_checked": 63660,
    "matched_rows": 63660,
    "missing_count": 0,
    "missing_examples": [],
    "ambiguous_count": 0,
    "ambiguous_examples": [],
    "many_to_one_mapping_count": 0,
    "unused_image_count": 0,
    "unused_image_examples": [],
    "examples_in_original_row_order": [
      {
        "original_id": "00001",
        "filename": "00001.jpg"
      },
      {
        "original_id": "00002",
        "filename": "00002.jpg"
      },
      {
        "original_id": "00003",
        "filename": "00003.jpg"
      }
    ]
  },
  "sample_subm

## 5. Save the data-access contract

Save counts, exact training classes, mapping rules and failed checks. Sample classes remain placeholders. No image contents, test class statistics, preprocessing, splits or models are used.

In [5]:
# Retain evidence of successful runs from both supported working directories.
previous = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}
run_key = "root" if Path.cwd().resolve() == PROJECT_ROOT else "notebooks"
execution = previous.get("execution", {})
execution[run_key] = {
    "working_directory": str(Path.cwd().resolve()),
    "python_executable": sys.executable,
    "kernel_name": "enge707-assessment3",
    "passed": not failed_checks,
}
summary = {
    "checked_at": datetime.now().astimezone().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "archive": archive_summary,
    "csv": csv_summary,
    "images": {
        "train_directory": relative(train_directory), "training_count": len(train_images),
        "test_directory": relative(test_directory), "test_count": len(test_images),
    },
    "training_classes": classes,
    "mapping_rules": {
        "train": "Five-digit original image_id + '.jpg'; exact match.",
        "sample_submission": "Unpadded original image_id.zfill(5) + '.jpg'; original ID unchanged.",
        "sample_classes": "Placeholders only; never treated as test labels.",
        "csv_handling": "dtype=str, keep_default_na=False, na_filter=False; original formatting and row order preserved.",
    },
    "mapping": mapping,
    "checks": checks,
    "failed_checks": failed_checks,
    "execution": execution,
}
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Saved:", relative(SUMMARY_PATH))
assert not failed_checks, f"Failed checks: {failed_checks}; inspect the JSON summary."
print("All data-access checks passed.")


Saved: outputs/metrics/data_access_check.json
All data-access checks passed.
